## Code for reading in DRAGONS data for Y. Uchibori's CHIME project
### A. Ordog, April 2025

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import healpy as hp

### Read in the data and errors (either galactic or equatorial)
#### Note: all files are in Healpix, with nside=64

In [ ]:
#######################################################
dir_files = '/home/aordog/Dropbox/DRAGONS_for_CHIME/'
coordsys = 'eq'
#coordsys = 'gal'
#######################################################

# The data
hduQ = fits.open(dir_files + 'Q_'+coordsys+'_K.hpx.fits')
hduU = fits.open(dir_files + 'U_'+coordsys+'_K.hpx.fits')
hduI = fits.open(dir_files + 'I_'+coordsys+'_K_offset_corr.hpx.fits')

Q = hduQ[0].data
U = hduU[0].data
I = hduI[0].data

# Errors in all data
hdudQ = fits.open(dir_files + 'dQ_'+coordsys+'_K.hpx.fits')
hdudU = fits.open(dir_files + 'dU_'+coordsys+'_K.hpx.fits')
hdudI = fits.open(dir_files + 'dI_'+coordsys+'_K.hpx.fits')

dQ = hdudQ[0].data
dU = hdudU[0].data
dI = hdudI[0].data

# PyGDSM model for comparison (saved on same grid)
hduPyGDMS  = fits.open(dir_files + 'pygdsm_eq_K.hpx.fits')
pygdsm = hduPyGDMS[0].data

# Header (all headers are the same for now, so just read in one)
hdr = hduI[0].header

### Check the header

In [ ]:
print(repr(hdr))

### Get frequency axis

In [ ]:
freqs = WCS(hdr).all_pix2world(0,range(I.shape[0]),0)[1]
print(freqs)

### Make coordinate axes

In [ ]:
all_pix = np.arange(0,hp.nside2npix(hdr['NSIDE']),1)
dec, ra = np.degrees(hp.pix2ang(nside=hdr['NSIDE'], ipix=all_pix))
dec = 90-dec

### Pick a frequency to look at

In [ ]:
##############
fpick = 602.0 # MHz
##############

fidx = np.abs(fpick - freqs/1e6).argmin()
print(freqs[fidx])

### Map of total intensity and error at chosen frequency

In [ ]:
hp.mollview(I[fidx],rot=(0,0),min=0,max=30, unit='K', 
            title='Stokes I '+str(np.round(freqs[fidx]/1e6,3))+' MHz')

hp.mollview(dI[fidx],rot=(0,0),min=0,max=5, unit='K', 
            title='error in Stokes I '+str(np.round(freqs[fidx]/1e6,3))+' MHz')


### Compare chosen frequency to PyGDSM model

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(14,5))
    
idx_good = np.where(np.isfinite(pygdsm[fidx]) & (np.isfinite(I[fidx])))[0]

ax[0].scatter(pygdsm[fidx][idx_good], I[fidx][idx_good], s=1)

h,xedges,yedges,im = ax[1].hist2d(pygdsm[fidx][idx_good], I[fidx][idx_good], bins=201, 
                   range=[[0,100], [0,100]],cmin=1,cmax=1000,cmap='cubehelix',norm='log')
cbar = plt.colorbar(im, ax=ax[1])
cbar.set_label('Number of data points')

for i in range(0,2):
    ax[i].set_xlim(0,100)
    ax[i].set_ylim(0,100)
    ax[i].grid(True)
    ax[i].set_aspect('equal')
    ax[i].set_xlabel(r'PyGDSM Stokes $I$ (K)')
    ax[i].set_ylabel(r'DRAGONS Stokes $I$ (K)')
    ax[i].set_title(r'$\nu$ = '+str(np.round(freqs[fidx]/1e6,3))+' MHz',fontsize=14)

### Map of Stokes Q or U and error at chosen frequency

In [ ]:
hp.mollview(U[fidx],rot=(0,0),min=-1,max=1, unit='K', cmap='RdBu_r',
            title='Stokes U '+str(np.round(freqs[fidx]/1e6,3))+' MHz')

hp.mollview(dU[fidx],rot=(0,0),min=0,max=0.5, unit='K', 
            title='error in Stokes U '+str(np.round(freqs[fidx]/1e6,3))+' MHz')


### Plot frequency spectrum at chosen point

In [ ]:
##########################
ra_test  = 40.0 # degrees
dec_test = 20.0 # degrees
##########################

idx = hp.pixelfunc.ang2pix(hdr['NSIDE'],np.radians(-dec_test+90.),np.radians(ra_test))
print(ra[idx],dec[idx])

In [ ]:
fig,ax = plt.subplots(3,1,figsize=(12,10))

ax[0].errorbar(freqs/1e6, Q[:,idx], yerr = dQ[:,idx],label='Stokes Q',marker='o',ms=1,mec='blue',mfc='blue')
ax[0].set_ylabel(r'Stokes $Q$ (K)')

ax[1].errorbar(freqs/1e6, U[:,idx], yerr = dU[:,idx],label='Stokes U',marker='o',ms=1,mec='blue',mfc='blue')
ax[1].set_ylabel(r'Stokes $U$ (K)')

ax[2].errorbar(freqs/1e6, I[:,idx], yerr = dI[:,idx],label='Stokes I, offset corrected',marker='o',ms=1,mec='blue',mfc='blue')
ax[2].set_ylabel(r'Stokes $I$ (K)')

for i in range(0,3):
    ax[i].set_xlim(350,1030)
    ax[i].set_ylim(-2,2)
    ax[i].legend(markerscale=3)
ax[2].set_ylim(0,40)
ax[2].set_xlabel('Frequency (MHz)')

### Beamwidths

In [ ]:
beam_arrays = np.genfromtxt(dir_files + 'dragons_beam_widths.txt')

fig,ax = plt.subplots(1,1,figsize=(8,5))
ax.scatter(beam_arrays[:,0], beam_arrays[:,1], s=5)

ax.set_xlim(350,1030)
ax.set_ylim(1,4)
ax.set_xlabel('Frequency (MHz)')
ax.set_ylabel('Beam FWHM (degrees)')
